In [1]:
from selenium import webdriver
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.firefox.firefox_profile import FirefoxProfile
from webdriver_manager.firefox import GeckoDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup
import pandas as pd
import time

# ---- CONFIGURACIÓN FIREFOX ----
profile_path = r"C:\Users\anali\AppData\Roaming\Mozilla\Firefox\Profiles\mrJjRiyd.Profile 1"

options = Options()
options.add_argument("-profile")
options.add_argument(profile_path)
options.add_argument("--headless")  # modo headless

profile = FirefoxProfile(profile_path)

driver = webdriver.Firefox(
    service=Service(GeckoDriverManager().install()),
    options=options
)

# ---- FUNCIÓN PARA OBTENER HTML RENDERIZADO ----
def obtener_html_renderizado(url: str) -> BeautifulSoup:
    driver.get(url)

    # Esperar que cargue al menos un artículo
    wait = WebDriverWait(driver,50)
    try:
        wait.until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "article.elementor_post")
            )
        )
    except TimeoutException:
        raise RuntimeError("La página cargó, pero los locales no aparecieron en el tiempo esperado")

    # ---- SCROLL AUTOMÁTICO HASTA EL FINAL ----
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(5)  # esperar carga dinámica
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

    html_renderizado = driver.page_source
    soup = BeautifulSoup(html_renderizado, "html.parser")
    return soup

# ---- FUNCIÓN PARA EXTRAER LOCALES Y CATEGORÍAS ----
def extraer_locales_y_categorias(soup) -> pd.DataFrame:
    data = []

    # Contenedor principal
    contenedor = soup.find("div", class_="page_content_wrap")
    if not contenedor:
        return pd.DataFrame(data)

    # Todos los artículos
    articulos = contenedor.find_all("article", class_="elementor_post")

    for art in articulos:
        # 1️⃣ Categoría desde la clase
        clases = art.get("class", [])
        categoria = None
        for cls in clases:
            if cls.startswith("cpt_portafolio_group-"):
                categoria = cls.replace("cpt_portafolio_group-", "")
                break

        # 2️⃣ Nombre del local
        div_text = art.find("div", class_="elementor-post__text")
        nombre = None
        if div_text:
            a_tag = div_text.find("a")
            if a_tag:
                nombre = a_tag.get_text(strip=True)

        if nombre and categoria:
            data.append({
                "LOCAL": nombre,
                "CATEGORIA": categoria
            })

    return pd.DataFrame(data)

# ---- EJECUTAR SCRAPER ----
url = "https://ccelrecreo.com/tiendas/"  # Página principal o sección de locales
soup = obtener_html_renderizado(url)
df_locales = extraer_locales_y_categorias(soup)

driver.quit()  # cerrar navegador

# Mostrar resultados
print(df_locales)


RuntimeError: La página cargó, pero los locales no aparecieron en el tiempo esperado

In [2]:
from selenium import webdriver
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options
from webdriver_manager.firefox import GeckoDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

options = Options()
# options.add_argument("--headless")  # desactivar mientras pruebas

driver = webdriver.Firefox(
    service=Service(GeckoDriverManager().install()),
    options=options
)

driver.get("https://ccelrecreo.com/tiendas/")

wait = WebDriverWait(driver, 20)

# XPath del contenedor que copiaste
xpath_contenedor = "/html/body/div[1]/div/div[3]/div/div/article/div/div/div[4]/div/div[2]/div/div/div[1]"

try:
    elem = wait.until(EC.presence_of_element_located((By.XPATH, xpath_contenedor)))
    
    # Buscamos el primer article dentro del contenedor usando XPath relativo
    primer_article = elem.find_element(By.XPATH, "./article[1]")
    clases = primer_article.get_attribute("class").split()
    grupo = None
    for cls in clases:
        if cls.startswith("cpt_portfolio_group-"):
            grupo = cls.replace("cpt_portfolio_group-", "")
            break

    print("Grupo/Categoría:", grupo)
        
except TimeoutException:
    print("No se encontró el elemento en el tiempo esperado")

driver.quit()


Grupo/Categoría: salud-y-belleza


In [4]:
from selenium import webdriver
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options
from webdriver_manager.firefox import GeckoDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup

# ---- CONFIGURACIÓN FIREFOX ----
options = Options()
# options.add_argument("--headless")  # desactivar mientras pruebas

driver = webdriver.Firefox(
    service=Service(GeckoDriverManager().install()),
    options=options
)

# ---- ABRIR PÁGINA ----
driver.get("https://ccelrecreo.com/tiendas/")
wait = WebDriverWait(driver, 20)

# ---- XPath del contenedor copiado ----
xpath_contenedor = "/html/body/div[1]/div/div[3]/div/div/article/div/div/div[4]/div/div[2]/div/div/div[1]"

try:
    # Esperar a que aparezca el contenedor
    elem = wait.until(EC.presence_of_element_located((By.XPATH, xpath_contenedor)))
    
    # Tomamos el primer article
    primer_article = elem.find_element(By.XPATH, "./article[1]")
    
    # --- 1️⃣ Extraer grupo/categoría ---
    clases = primer_article.get_attribute("class").split()
    grupo = None
    for cls in clases:
        if cls.startswith("cpt_portfolio_group-"):
            grupo = cls.replace("cpt_portfolio_group-", "")
            break
    print("Grupo/Categoría:", grupo)
    
    # --- 2️⃣ Extraer nombre del local ---
    soup = BeautifulSoup(primer_article.get_attribute("outerHTML"), "html.parser")
    div_text = soup.find("div", class_="elementor-post__text")
    nombre_local = None
    if div_text:
        a_tag = div_text.find("a")
        if a_tag:
            nombre_local = a_tag.get_text(strip=True)
    print("Nombre del local:", nombre_local)
        
except TimeoutException:
    print("No se encontró el elemento en el tiempo esperado")

driver.quit()


Grupo/Categoría: salud-y-belleza
Nombre del local: FUNKY FISH


In [9]:
from selenium import webdriver
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options
from webdriver_manager.firefox import GeckoDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup
import time
import pandas as pd

# ---- CONFIGURACIÓN FIREFOX ----
options = Options()
# options.add_argument("--headless")  # desactivar mientras pruebas

driver = webdriver.Firefox(
    service=Service(GeckoDriverManager().install()),
    options=options
)

# ---- FUNCIÓN PARA OBTENER HTML RENDERIZADO CON SCROLL ----
def obtener_html_renderizado(url: str) -> BeautifulSoup:
    driver.get(url)
    wait = WebDriverWait(driver, 50)
    
    # XPath del contenedor principal de artículos
    xpath_contenedor = "/html/body/div[1]/div/div[3]/div/div/article/div/div/div[4]/div/div[2]/div/div/div[1]"
    
    try:
        # Esperar que aparezca el contenedor
        wait.until(
            EC.presence_of_element_located((By.XPATH, xpath_contenedor))
        )
    except TimeoutException:
        raise RuntimeError("La página cargó, pero no apareció el contenedor de locales")
    
    # ---- SCROLL AUTOMÁTICO HASTA EL FINAL ----
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(10)  # espera que cargue contenido dinámico
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

    # Obtener HTML final renderizado
    html_renderizado = driver.page_source
    soup = BeautifulSoup(html_renderizado, "html.parser")
    return soup, xpath_contenedor

# ---- FUNCIÓN PARA EXTRAER LOCALES Y CATEGORÍAS ----
def extraer_locales_y_categorias(soup: BeautifulSoup, xpath_contenedor: str) -> pd.DataFrame:
    # Buscamos el contenedor con Selenium
    elem = driver.find_element(By.XPATH, xpath_contenedor)
    
    # Todos los <article> dentro del contenedor
    articulos = elem.find_elements(By.XPATH, "./article")
    
    data = []
    for art in articulos:
        # ---- Grupo/Categoría ----
        clases = art.get_attribute("class").split()
        grupo = None
        for cls in clases:
            if cls.startswith("cpt_portfolio_group-"):
                grupo = cls.replace("cpt_portfolio_group-", "")
                break
        
        # ---- Nombre del local ----
        soup_art = BeautifulSoup(art.get_attribute("outerHTML"), "html.parser")
        div_text = soup_art.find("div", class_="elementor-post__text")
        nombre_local = None
        if div_text:
            a_tag = div_text.find("a")
            if a_tag:
                nombre_local = a_tag.get_text(strip=True)
        
        if nombre_local and grupo:
            data.append({
                "LOCAL": nombre_local,
                "CATEGORIA": grupo
            })
    
    return pd.DataFrame(data)

# ---- EJECUTAR SCRAPER ----
url = "https://ccelrecreo.com/tiendas/"
soup, xpath_contenedor = obtener_html_renderizado(url)
df_locales = extraer_locales_y_categorias(soup, xpath_contenedor)

print(df_locales)

driver.quit()


                           LOCAL        CATEGORIA
0                     FUNKY FISH  salud-y-belleza
1                        REMY EC  salud-y-belleza
2               ANTOJITOS EC 593      gastronomia
3    TARTA VASCA DE LA CHEF CARO      gastronomia
4                            LEE             moda
..                           ...              ...
536       CROISSANT LOVER’S CAFÉ      gastronomia
537                       CONECA        vehiculos
538                   CLARO ISLA        servicios
539                        ANETA        servicios
540                     Explorer             moda

[541 rows x 2 columns]


In [10]:
df_locales.sort_values(by = 'CATEGORIA').to_excel(r"ccrecreo.xlsx")